In [ ]:
import pandas as pd
import fastf1
import os

In [ ]:
#folder za kes
cache_path = 'temp_cache'
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
fastf1.set_log_level('ERROR') #da se smanji ispis fastf1 biblioteke

In [ ]:
# ucitava se csv
df = pd.read_csv('tables/all_races.csv')
print(f"Sve trke u CSV: {len(df)}")

# uzimaju se samo neki podaci
df = df[['season', 'round', 'race_name', 'date', 'time']]
df.columns = ['season', 'round', 'race_name', 'date', 'time']
df = df[df['season'] >= 2018]  # samo posle 2018, za pre ne postoje podaci
print(f"Trke posle 2018 : {len(df)}")

In [ ]:
# Ucitavanje podaci koji vec postoje
processed_file = 'tables/all_weather.csv'
if os.path.exists(processed_file):
    df_done = pd.read_csv(processed_file)
    completed_set = set(zip(df_done['season'], df_done['round']))
    print(f"Vec postojece: {len(completed_set)}")
else:
    df_done = pd.DataFrame()
    completed_set = set()
    print("Ne postoji nista")

In [ ]:
# Ucitavanje novih podataka
new_rows = []

for idx, row in df.iterrows():
    season = int(row['season'])
    round_num = int(row['round'])

    if (season, round_num) in completed_set:
        print(f"Preskace se {season} runda {round_num}: vec postoji")
        continue

    try:
        print(f"Ucitavanje {season} runda {round_num}...")
        session = fastf1.get_session(season, round_num, 'R')
        session.load(telemetry=False, weather=True)

        weather_df = session.weather_data
        if weather_df.empty:
            print(f"Ne postoje podaci za vreme za {season} runda {round_num}")
            continue

        mean_weather = weather_df.mean(numeric_only=True).to_dict()
        enriched_row = row.to_dict()
        enriched_row.update(mean_weather)
        new_rows.append(enriched_row)
        print(f"Dodato vreme za {season} runda {round_num}")

    except Exception as e:
        print(f"Neupesno za {season} runda {round_num}: {e}")

In [ ]:
# Kombinovanje sa starim i cuvanje
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_done, df_new], ignore_index=True)
    df_combined.to_csv(processed_file, index=False)
    print(f"Sacuvano {len(df_new)} novih redova {processed_file}")
else:
    print("Nista nije dodato")
